# Step 3. Write, and read back

*With ngff-zarr.*

**Goal:** write both images and the transformation between them in one OME-Zarr 0.6 file, and check what a reader finds in it. No registration tool is needed from here on.

In [1]:
import json
import shutil
import zipfile
from pathlib import Path

import ngff_zarr as nz
import numpy as np
from ngff_zarr import CoordinateSystemIdentifier, NgffScene
from ngff_zarr.v06.zarr_metadata import Displacements

FIXED, MOVING = "../output/R0.ome.zarr", "../output/R1.ome.zarr"
TRANSFORM = "../output/registration/P000/Transform.ome.zarr"
STORE, SCENE = "../output/pair.ome.zarr", "../output/pair.ozx"
FIELD = "coordinateTransformations/registration"

shutil.rmtree(STORE, ignore_errors=True)  # so that this cell can run twice

# The transformation is an OME-Zarr image already: it goes below the scene, first.
nz.to_ome_zarr(f"{STORE}/{FIELD}", nz.from_ome_zarr(TRANSFORM), version="0.6")

# The scene names what it connects: from the coordinates of the fixed image to those of the moving one.
registration = Displacements(
    path=FIELD,
    interpolation="linear",
    input=CoordinateSystemIdentifier(path="fixed", name="intrinsic"),
    output=CoordinateSystemIdentifier(path="moving", name="intrinsic"),
)
images = {"fixed": nz.from_ome_zarr(FIXED), "moving": nz.from_ome_zarr(MOVING)}
NgffScene(images=images, coordinateTransformations=[registration]).to_ome_zarr(STORE, overwrite=False)

# One file: the same store, zipped (RFC-9).
nz.write_store_to_zip(STORE, SCENE, version="0.6")
print(f"{SCENE}: {Path(SCENE).stat().st_size / 1e6:.0f} MB")

../output/pair.ozx: 141 MB


`pair.ozx` is one file:

```text
pair.ozx
├── zarr.json                   the scene: what the transformation connects
├── fixed/                      R0, with its pyramid
├── moving/                     R1, with its pyramid
└── coordinateTransformations/
    └── registration/           the field of step 2
```

- **The transformation was OME-Zarr already.** Step 2 wrote it as an RFC-5 `displacements` field. It is copied as it is: an ordinary OME-Zarr image, whose first axis holds the components (dz, dy, dx), in micrometers.
- **The scene says what it connects:** from the coordinate system of `fixed` to the one of `moving`.

## Read it back

This is what a reader finds: the file's metadata, plus one vector of the field.

In [2]:
scene = NgffScene.from_ome_zarr(SCENE)
for name, multiscales in scene.images.items():
    size = " x ".join(str(extent) for extent in multiscales.images[0].data.shape)
    print(f"{name}: {size} voxels, {len(multiscales.images)} levels")

transformation = scene.coordinateTransformations[0]
print(f"transformation: {transformation.type}, from {transformation.input.path} to {transformation.output.path}")

field = nz.from_ome_zarr(f"{SCENE}/{transformation.path}").images[0]
print(f"its field: {transformation.path}, {' x '.join(str(extent) for extent in field.data.shape)}")

centre = tuple(extent // 2 for extent in field.data.shape[1:])
vector = np.asarray(field.data[(slice(None), *centre)])
moves = ", ".join(f"d{axis} {value:.1f}" for axis, value in zip(field.dims[1:], vector))
print(f"at the centre of the image: {moves} {field.axes_units['x']}s")

fixed: 50 x 680 x 535 voxels, 3 levels
moving: 50 x 680 x 535 voxels, 3 levels
transformation: displacements, from fixed to moving
its field: coordinateTransformations/registration, 3 x 50 x 680 x 535
at the centre of the image: dz 1.3, dy -9.8, dx 36.4 micrometers


The last line reads directly: how far the centre of the organoid moved between the two rounds, in micrometers.

Any OME-Zarr 0.6 reader finds the same thing in `zarr.json`, at the root of the file:

In [3]:
with zipfile.ZipFile(SCENE) as archive:
    root = json.loads(archive.read("zarr.json"))
print(json.dumps(root["attributes"]["ome"]["scene"]["coordinateTransformations"], indent=2))

[
  {
    "input": {
      "path": "fixed",
      "name": "intrinsic"
    },
    "output": {
      "path": "moving",
      "name": "intrinsic"
    },
    "type": "displacements",
    "path": "coordinateTransformations/registration",
    "interpolation": "linear"
  }
]


## The size of the transformation

| Preset | `Transform.h5` | `Transform.ome.zarr` | `pair.ozx` |
|---|---|---|---|
| `Generic_Rigid` | 437 MB | 95 MB | 141 MB |
| `ConvexAdam_Composite` | 437 MB | 170 MB | 217 MB |

The field is stored as float32, on the fixed image's grid.

**Next: [Step 4. Apply](4-apply.ipynb)**